# Block 2C: Computer Vision — Car Damage Classification

**Goal:** Train a classifier to detect and categorize car damage types from images.

**Dataset:** VehiDE — 13,945 high-resolution images, 8 damage categories (Kaggle, no access request needed)

**Model:** EfficientNet-B0 (fine-tuned) vs. ResNet-18 (baseline)

**Output:** Damage type + confidence score → used as input feature for ML block

## Setup

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
import timm
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

DATA_DIR = Path('../data/raw/vehide')
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
PROCESSED_DIR.mkdir(exist_ok=True)

## Data Source

**VehiDE Dataset** — [https://www.kaggle.com/datasets/hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection](https://www.kaggle.com/datasets/hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection)

- 13,945 high-resolution car damage images
- 8 damage categories: `dents`, `scratch`, `broken_glass`, `broken_lights`, `lost_parts`, `torn`, `punctured`, `non_damaged`
- Annotations provided in an Excel file (`annotations.xlsx`) with columns: `filename`, `damage_class`
- Publicly available on Kaggle — no email request required

**Download:**
```bash
kaggle datasets download -d hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection -p data/raw/
cd data/raw && unzip vehide-dataset-automatic-vehicle-damage-detection.zip -d vehide/
```

**Expected structure:**
```
data/raw/vehide/
  images/
    img_001.jpg
    img_002.jpg
    ...
  annotations.xlsx
```

## Exploratory Data Analysis (EDA)

In [ ]:
DAMAGE_CLASSES = ['dents', 'scratch', 'broken_glass', 'broken_lights', 'lost_parts', 'torn', 'punctured', 'non_damaged']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(DAMAGE_CLASSES)}

# Load annotations from Excel
df_ann = pd.read_excel(DATA_DIR / 'annotations.xlsx')
print(f'Annotation columns: {df_ann.columns.tolist()}')
print(f'Total samples: {len(df_ann)}')

# Normalize column names (adapt if VehiDE uses different column names)
df_ann.columns = df_ann.columns.str.strip().str.lower()
filename_col = [c for c in df_ann.columns if 'file' in c or 'image' in c or 'name' in c][0]
label_col    = [c for c in df_ann.columns if 'class' in c or 'label' in c or 'damage' in c][0]
df_ann = df_ann.rename(columns={filename_col: 'filename', label_col: 'damage_class'})

# Keep only known classes
df_ann['damage_class'] = df_ann['damage_class'].str.strip().str.lower().str.replace(' ', '_')
df_ann = df_ann[df_ann['damage_class'].isin(DAMAGE_CLASSES)].reset_index(drop=True)
df_ann['label'] = df_ann['damage_class'].map(CLASS_TO_IDX)

print(f'\nClass distribution:')
print(df_ann['damage_class'].value_counts())

# Train / Val / Test split (70/15/15)
df_train, df_temp = train_test_split(df_ann, test_size=0.30, stratify=df_ann['label'], random_state=SEED)
df_val, df_test   = train_test_split(df_temp, test_size=0.50, stratify=df_temp['label'], random_state=SEED)
print(f'\nSplits — Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df_ann['damage_class'].value_counts()
counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Image Count per Damage Class')
axes[0].set_xlabel('Damage Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution (Total)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'cv_class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Show sample images per class
n_classes = len(DAMAGE_CLASSES)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, cls in enumerate(DAMAGE_CLASSES):
    sample = df_ann[df_ann['damage_class'] == cls].iloc[0]
    img_path = DATA_DIR / 'images' / sample['filename']
    if img_path.exists():
        img = Image.open(img_path).convert('RGB')
        axes[i].imshow(img)
    axes[i].set_title(f'{cls}\n({counts.get(cls, 0)} images)')
    axes[i].axis('off')

plt.suptitle('Sample Images per Damage Class — VehiDE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'cv_sample_images.png', dpi=150)
plt.show()

In [ ]:
# Analyze image dimensions
widths, heights = [], []
for cls in DAMAGE_CLASSES:
    for img_path in list((DATA_DIR / 'train' / cls).glob('*.jpg'))[:20]:
        img = Image.open(img_path)
        widths.append(img.width)
        heights.append(img.height)

print(f'Width  — mean: {np.mean(widths):.0f}, min: {min(widths)}, max: {max(widths)}')
print(f'Height — mean: {np.mean(heights):.0f}, min: {min(heights)}, max: {max(heights)}')

## Image Preprocessing & Augmentation

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
IMG_DIR = DATA_DIR / 'images'

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class VehiDEDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / row['filename']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])

train_dataset = VehiDEDataset(df_train, IMG_DIR, train_transform)
val_dataset   = VehiDEDataset(df_val,   IMG_DIR, val_transform)
test_dataset  = VehiDEDataset(df_test,  IMG_DIR, val_transform)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# Weighted sampler to handle class imbalance
class_counts = Counter([label for _, label in train_dataset.samples])
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for _, label in train_dataset.samples]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Model 1: ResNet-18 (Baseline)

In [ ]:
def build_resnet18(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, model_name='model'):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = eval_epoch(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch+1:02d}/{epochs} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODELS_DIR / f'{model_name}_best.pth')

    print(f'\nBest Val Accuracy: {best_val_acc:.4f}')
    return history

In [ ]:
NUM_CLASSES = len(DAMAGE_CLASSES)
resnet18 = build_resnet18(NUM_CLASSES)
print(f'ResNet-18 parameters: {sum(p.numel() for p in resnet18.parameters()):,}')

history_resnet = train_model(resnet18, train_loader, val_loader, epochs=15, model_name='resnet18')

## Model 2: EfficientNet-B0 (Primary Model)

In [ ]:
def build_efficientnet_b0(num_classes):
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
    return model

efficientnet = build_efficientnet_b0(NUM_CLASSES)
print(f'EfficientNet-B0 parameters: {sum(p.numel() for p in efficientnet.parameters()):,}')

history_effnet = train_model(efficientnet, train_loader, val_loader, epochs=15, model_name='efficientnet_b0')

## Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_resnet['val_acc'], label='ResNet-18', color='orange')
axes[0].plot(history_effnet['val_acc'], label='EfficientNet-B0', color='steelblue')
axes[0].set_title('Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_resnet['val_loss'], label='ResNet-18', color='orange')
axes[1].plot(history_effnet['val_loss'], label='EfficientNet-B0', color='steelblue')
axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('ResNet-18 vs EfficientNet-B0 Training', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'cv_model_comparison.png', dpi=150)
plt.show()

## Evaluation on Test Set

In [ ]:
def evaluate_on_test(model, model_path, loader):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

preds_resnet, labels, _ = evaluate_on_test(resnet18, MODELS_DIR / 'resnet18_best.pth', test_loader)
preds_effnet, _, probs_effnet = evaluate_on_test(efficientnet, MODELS_DIR / 'efficientnet_b0_best.pth', test_loader)

print('=== ResNet-18 ===')
print(classification_report(labels, preds_resnet, target_names=DAMAGE_CLASSES))

print('\n=== EfficientNet-B0 ===')
print(classification_report(labels, preds_effnet, target_names=DAMAGE_CLASSES))

In [ ]:
# Confusion matrix for best model (EfficientNet)
cm = confusion_matrix(labels, preds_effnet)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=DAMAGE_CLASSES, yticklabels=DAMAGE_CLASSES)
plt.title('Confusion Matrix — EfficientNet-B0 (Test Set)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'cv_confusion_matrix.png', dpi=150)
plt.show()

## Export for ML Block Integration

The CV model outputs (damage class + confidence scores) are saved as structured features
for use in the ML block's repair cost estimation model.

In [ ]:
# Save predictions + confidence scores for ML block
cv_output = pd.DataFrame(probs_effnet, columns=[f'cv_prob_{cls}' for cls in DAMAGE_CLASSES])
cv_output['cv_pred_class'] = [DAMAGE_CLASSES[p] for p in preds_effnet]
cv_output['cv_pred_idx'] = preds_effnet
cv_output['cv_confidence'] = probs_effnet.max(axis=1)
cv_output['true_label'] = [DAMAGE_CLASSES[l] for l in labels]

cv_output.to_csv(PROCESSED_DIR / 'cv_predictions.csv', index=False)
print(f'Saved {len(cv_output)} CV predictions to processed/cv_predictions.csv')
cv_output.head()

In [ ]:
# Save class mapping for inference
class_mapping = {i: cls for i, cls in enumerate(DAMAGE_CLASSES)}
with open(MODELS_DIR / 'cv_class_mapping.json', 'w') as f:
    json.dump(class_mapping, f)

print('Saved class mapping:', class_mapping)

## Summary

| Model | Test Accuracy | Params |
|---|---|---|
| ResNet-18 | — | 11.7M |
| EfficientNet-B0 | — | 5.3M |

**EfficientNet-B0** was selected as the primary model due to higher accuracy with fewer parameters.

**Integration with ML block:** The predicted damage class and per-class confidence scores 
are passed as features to the repair cost estimation model in `02_ml_training.ipynb`.